# Reranker trade-offs: candidates, windows and speed (Colab GPU)

Follows `reranker_colab.ipynb`, which showed that the blended MaxP reranker (`bge-reranker-v2-m3`,
2,000-character windows, 512 tokens, reranker weight 0.7) raises development Recall@10 from 0.722
to 0.773. This run compares three settings of that reranker in **one pass** over the 300
development questions, each with blend weights 1, 0.9, 0.8, 0.7 and 0.6 and a cross-validated
(`blendcv`) result:

| Setting | Candidates reranked | Windows per document | Question it answers |
| --- | ---: | ---: | --- |
| Current | 50 (in practice BM25's top 50) | 8 | Baseline, now with weights 0.8 and 0.9 |
| Deep | 100 (BM25's 50 + the dense model's 50) | 8 | Do dense-only candidates help now? |
| Fast | 20 (the hybrid's top 20) | 4 | How much accuracy does a ~5x cheaper setting lose? |

Searches are run once. Scores are cached per document, so the deep setting reuses the current
setting's scores, and every blend weight reuses them all. Reranking cost per setting is
estimated from the windows it scores per question times the measured seconds per window.

1. **Runtime → Change runtime type → L4 (or T4) GPU, with High-RAM on.**
2. **Part 4** compares the settings and recommends one under `MAX_ADDED_SECONDS` (GPU seconds
   added per question), if you set it. No calibration run: calibration has been used twice.
3. Then time the chosen setting on your own computer with `scripts/time_reranker.py`.

In [ ]:
!nvidia-smi --query-gpu=name,memory.total --format=csv
!free -g | head -2

## 1. Drive and project code

Set `WORK` to the Drive folder that holds the data. The defaults match the personal `rag-dense`
layout (`dense_base_titled` at its top level). For the team `files_web_app` folder, use
`DENSE = f"{WORK}/indexes/dense-bge-base"`.

In [ ]:
import json
import os
from google.colab import drive

drive.mount("/content/drive")
WORK = "/content/drive/MyDrive/rag-dense"
CORPUS = f"{WORK}/documents.parquet"
QUESTIONS = f"{WORK}/questions.jsonl"
DRIVE_BM25 = f"{WORK}/indexes/bm25"
DENSE = f"{WORK}/dense_base_titled"  # BGE-base, title on every passage
BM25 = "/content/indexes/bm25"  # Local copy: much faster than reading from Drive.

REPO = "https://github.com/AgenticEnterpriseSystem/Agentic-Enterprise-RAG-System.git"
BRANCH = "hira/dev"
# For a private repository, clone with a GitHub token instead (it is not displayed):
# from getpass import getpass; REPO = f"https://{getpass('GitHub token: ')}@github.com/AgenticEnterpriseSystem/Agentic-Enterprise-RAG-System.git"
if not os.path.exists("/content/project"):
    !git clone --depth 1 -b {BRANCH} {REPO} /content/project
%cd /content/project
!git log -1 --format='%h %s'
!pip install -q -r requirements.txt

## 2. Check the indexes

Both indexes must come from the same `documents.parquet`. If the BM25+ index on Drive is missing
or was built from a different copy, it is rebuilt here (about 5 minutes).

In [ ]:
for path in (CORPUS, QUESTIONS, f"{DENSE}/manifest.json"):
    assert os.path.exists(path), f"Missing {path}"
dense_manifest = json.load(open(f"{DENSE}/manifest.json"))
assert dense_manifest.get("complete"), "The dense index is incomplete"
from src.retrieval.corpus_dense import model_label
LABEL = model_label(dense_manifest["model"]["name"])
print("Dense model:", LABEL, dense_manifest["model"]["revision"])

def bm25_matches(path):
    manifest = f"{path}/manifest.json"
    return os.path.exists(manifest) and json.load(open(manifest))["corpus_sha256"] == dense_manifest["corpus_sha256"]

if not bm25_matches(BM25):
    if bm25_matches(DRIVE_BM25):
        print("Copying the BM25+ index from Drive...")
        !rm -rf {BM25} && mkdir -p /content/indexes && cp -r {DRIVE_BM25} {BM25}
    else:
        print("Building a BM25+ index from the same corpus (about 5 minutes)...")
        !python -m src.retrieval.corpus_bm25 --corpus {CORPUS} --output {BM25}
assert bm25_matches(BM25)
print("Indexes ready.")

## 3. Development run

One command scores all three settings. Expect roughly 15–40 minutes on an L4.

In [ ]:
RERANKER = "bge-reranker-v2-m3"
DEPTHS = "50:8,100:8,20:4"  # candidates:windows per document
COMMON = (f"--bm25-index {BM25} --dense-index {DENSE} --corpus {CORPUS} --questions {QUESTIONS} "
          f"--device cuda --bm25-weights 0.7 --rerank-bm25-weight 0.7 --rerankers {RERANKER} "
          f"--rerank-window-chars 2000 --reranker-max-length 512 --reranker-batch-size 32")
!python -m src.evaluation.run_hybrid {COMMON} --split development \
    --rerank-weights 1,0.9,0.8,0.7,0.6 --rerank-depths {DEPTHS}

## 4. Compare the settings

Each setting is judged by its cross-validated (`blendcv`) result. With `MAX_ADDED_SECONDS` set,
the recommendation is the cheapest setting within that budget that is not clearly worse than the
most accurate one (its paired Recall@10 interval against the best still reaches zero).

In [ ]:
import glob
from src.evaluation.retrieval_metrics import evaluate_retrieval, paired_bootstrap, recall_at_k, ndcg_at_k

MAX_ADDED_SECONDS = None  # e.g. 3.0 GPU seconds added per question; None = report only

def load_rows(path):
    return [json.loads(line) for line in open(f"{path}/predictions.jsonl")]

costs = json.load(open("evaluation/results/reranker_costs_development.json"))
base = load_rows(f"evaluation/results/hybrid_{LABEL}_w0.7_full_development")
o = evaluate_retrieval(base)["overall"]
print(f"Hybrid without a reranker: R@10 {o['recall@10']['mean']:.3f}, nDCG@10 {o['ndcg@10']['mean']:.3f}\n")
settings = {}
for stem, cost in costs.items():
    rows = load_rows(f"evaluation/results/{stem}_blendcv_full_development")
    o = evaluate_retrieval(rows)["overall"]
    r = paired_bootstrap(rows, base, metric=recall_at_k)
    n = paired_bootstrap(rows, base, metric=ndcg_at_k)
    chosen = json.load(open(f"evaluation/results/{stem}_blendcv_full_development/summary.json"))["cross_validation"]["chosen_per_fold"]
    settings[stem] = dict(rows=rows, recall=o["recall@10"]["mean"], ndcg=o["ndcg@10"]["mean"], cost=cost, weights=chosen)
    print(f"{stem}\n   {cost['candidates']} candidates, {cost['windows_per_question']:.0f} windows/question, "
          f"~{cost['seconds_per_question']:.2f}s added per question (GPU)\n"
          f"   R@10 {o['recall@10']['mean']:.3f} ({r['change']:+.3f}, {r['low']:+.3f} to {r['high']:+.3f}; "
          f"{r['better']} better, {r['worse']} worse)   nDCG@10 {o['ndcg@10']['mean']:.3f} "
          f"({n['change']:+.3f}, {n['low']:+.3f} to {n['high']:+.3f})   weights per fold {chosen}\n")

best = max(settings, key=lambda s: (settings[s]["recall"], settings[s]["ndcg"]))
print("Most accurate:", best)
if MAX_ADDED_SECONDS is not None:
    affordable = [s for s in settings if settings[s]["cost"]["seconds_per_question"] <= MAX_ADDED_SECONDS]
    close = [s for s in affordable if s == best or
             paired_bootstrap(settings[s]["rows"], settings[best]["rows"], metric=recall_at_k)["high"] >= 0]
    pick = min(close, key=lambda s: settings[s]["cost"]["seconds_per_question"]) if close else None
    print(f"Recommended under {MAX_ADDED_SECONDS}s:", pick or "none; raise the budget or use the fast setting")

## 5. Save the results

Downloads `reranker_tradeoffs_results.zip`. Upload it to add the results to the repository.
Then time the chosen setting on your own computer, for example the fast one:

```sh
python scripts/time_reranker.py --count 20 --rerank-k 20 --rerank-max-windows 4
```

In [ ]:
KEEP = f"hybrid_{LABEL}_w0.7_{RERANKER}_maxp2000* hybrid_{LABEL}_w0.7_full_development reranker_costs_development.json"
!mkdir -p {WORK}/results
!cd evaluation/results && cp -r {KEEP} {WORK}/results/ 2>/dev/null; true
!cd evaluation/results && zip -qr /content/reranker_tradeoffs_results.zip {KEEP} 2>/dev/null; true
from google.colab import files
files.download("/content/reranker_tradeoffs_results.zip")